# Chapter 2.6: split revenue growth into volume and price mix

Synthetic Kestrel data, two financial years, by route. Price mix = (1 + revenue growth) / (1 + volume growth) - 1.
Look at whether each route grows in hectolitres or in rupees a hectolitre. Every figure is synthetic.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
from calculator import inr, r2

In [ ]:
v = pd.read_csv('../data/volume_by_year.csv')
tot = v.groupby('fy')[['hl', 'revenue_inr']].sum()
vol_growth = tot.hl['FY2027'] / tot.hl['FY2026'] - 1
rev_growth = tot.revenue_inr['FY2027'] / tot.revenue_inr['FY2026'] - 1
price_mix = (1 + rev_growth) / (1 + vol_growth) - 1
print('FY2026:', inr(tot.hl['FY2026'], 0), 'hL; revenue INR', inr(tot.revenue_inr['FY2026'], 0))
print(f'volume {vol_growth:.1%}, revenue {rev_growth:.1%}, price mix {price_mix:.1%}')

The same split by route. Route A is the private wholesaler (30 days); Route C the corporation (60 days).

In [ ]:
r = v.groupby(['route', 'fy'])[['hl', 'revenue_inr']].sum().unstack('fy')
split = pd.DataFrame({
    'volume growth': r.hl.FY2027 / r.hl.FY2026 - 1,
    'revenue growth': r.revenue_inr.FY2027 / r.revenue_inr.FY2026 - 1})
split['price mix'] = (1 + split['revenue growth']) / (1 + split['volume growth']) - 1
route_c_share = r.hl.FY2026['Route C'] / tot.hl['FY2026']
print(split.round(3))
print(f'Route C carries {route_c_share:.0%} of FY2026 volume and waits twice as long for its cash')

## The bands of chapter 2.2
The author's bands against a synthetic popular shelf price of INR 190 a 650 ml bottle.

In [ ]:
anchor = 190
edges = {'economy': 0.9 * anchor, 'popular': 1.1 * anchor, 'premium': 1.5 * anchor}
def band(mrp):
    x = mrp / anchor
    return 'economy' if x < 0.9 else 'popular' if x <= 1.1 else 'premium' if x <= 1.5 else 'super premium'
lager_ratio = 220 / anchor
print(f'Kestrel Lager at 220: {lager_ratio:.2f} x, {band(220)}; stout at 265: {band(265)}')
mix5 = 0.05 * (265 - anchor)
mix20 = 0.20 * (265 - anchor)
print(f'5 points to the stout: +INR {mix5:.2f} ({mix5 / anchor:.1%}); 20 points: +INR {mix20:.2f} ({mix20 / anchor:.1%})')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.2))
for (lo, hi, name) in [(140, 171, 'Economy'), (171, 209, 'Popular'), (209, 285, 'Premium'), (285, 320, 'Super premium')]:
    ax.axvspan(lo, hi, alpha=0.25 if name in ('Popular', 'Super premium') else 0.1)
    ax.text((lo + hi) / 2, 0.5, name, ha='center')
for x, label in [(220, 'Kestrel Lager 220'), (265, 'Stout 265')]:
    ax.axvline(x, color='k'); ax.text(x, 0.85, label, ha='center')
ax.set_yticks([]); ax.set_xlabel('MRP, INR a bottle (synthetic State S)')
plt.show()